In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder.master("local[*]").appName("weather_preparation").getOrCreate()
base_path = "/home/jovyan/work"

In [5]:
df_raw = spark.read.option("header", True).csv(f"{base_path}/data/weather/csv/")

df_raw.printSchema()

root
 |-- STATION: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- ELEVATION: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- REPORT_TYPE: string (nullable = true)
 |-- SOURCE: string (nullable = true)
 |-- HourlyAltimeterSetting: string (nullable = true)
 |-- HourlyDewPointTemperature: string (nullable = true)
 |-- HourlyDryBulbTemperature: string (nullable = true)
 |-- HourlyPrecipitation: string (nullable = true)
 |-- HourlyPresentWeatherType: string (nullable = true)
 |-- HourlyPressureChange: string (nullable = true)
 |-- HourlyPressureTendency: string (nullable = true)
 |-- HourlyRelativeHumidity: string (nullable = true)
 |-- HourlySkyConditions: string (nullable = true)
 |-- HourlySeaLevelPressure: string (nullable = true)
 |-- HourlyStationPressure: string (nullable = true)
 |-- HourlyVisibility: string (nullable = true)
 |-- HourlyWetBulbTemperature: string (nu

In [8]:
df_raw.write.parquet(f"{base_path}/data/weather/raw_weather/")

In [3]:
df_raw = spark.read.parquet(f"{base_path}/data/weather/raw_weather/")

In [4]:
print(df_raw.count())

1119758


In [4]:
df_raw.select("REPORT_TYPE").distinct().show()

+-----------+
|REPORT_TYPE|
+-----------+
|        SOM|
|      FM-15|
|        SOD|
|      FM-12|
|      FM-16|
|       FM13|
|       NULL|
+-----------+



In [51]:
df_raw.filter(df_raw.REPORT_TYPE == "FM-15").filter(df_raw.HourlyPrecipitation.isNotNull()).select("DATE","REPORT_TYPE", "DailyPrecipitation", "DailyWeather", "HourlyPrecipitation", "HourlyPresentWeatherType").show()

+-------------------+-----------+------------------+------------+-------------------+------------------------+
|               DATE|REPORT_TYPE|DailyPrecipitation|DailyWeather|HourlyPrecipitation|HourlyPresentWeatherType|
+-------------------+-----------+------------------+------------+-------------------+------------------------+
|2017-01-02T03:35:00|      FM-15|              NULL|        NULL|                1.0|                    NULL|
|2017-01-02T03:55:00|      FM-15|              NULL|        NULL|                1.0|                    NULL|
|2017-01-02T03:56:00|      FM-15|              NULL|        NULL|                1.0|                    NULL|
|2017-01-02T04:15:00|      FM-15|              NULL|        NULL|              252.9|                    NULL|
|2017-01-02T04:35:00|      FM-15|              NULL|        NULL|              252.9|                    NULL|
|2017-01-02T04:55:00|      FM-15|              NULL|        NULL|              252.9|                    NULL|
|

In [21]:
df_filtered = df_raw \
    .filter(col("REPORT_TYPE") == "FM-15") \
    .filter(year(to_timestamp(col("DATE"))).between(2023, 2025)) \
    .select("STATION", "NAME", "LATITUDE", "LONGITUDE", "DATE", "HourlyDryBulbTemperature", "HourlyPrecipitation", "HourlyPresentWeatherType") \
    .withColumn("DATE", to_timestamp(col("DATE"))) \
    .withColumn("DAY", to_date(col("DATE")))

In [22]:
df_day = df_raw \
    .filter(df_raw.REPORT_TYPE == "SOD") \
    .select("STATION", "DATE", "DailyAverageDryBulbTemperature", "DailyMaximumDryBulbTemperature", "DailyMinimumDryBulbTemperature", "DailyPrecipitation", "DailyWeather", "Sunrise", "Sunset") \
    .withColumn("DAY", to_date(col("DATE"))) \
    .drop("DATE")

In [23]:
df_filtered = df_filtered \
    .join(df_day, on=["STATION", "DAY"], how="inner") \
    .drop("DAY")

In [24]:
df_filtered.show(5)

+-----------+--------------------+--------+---------+-------------------+------------------------+-------------------+------------------------+------------------------------+------------------------------+------------------------------+------------------+------------+-------+------+
|    STATION|                NAME|LATITUDE|LONGITUDE|               DATE|HourlyDryBulbTemperature|HourlyPrecipitation|HourlyPresentWeatherType|DailyAverageDryBulbTemperature|DailyMaximumDryBulbTemperature|DailyMinimumDryBulbTemperature|DailyPrecipitation|DailyWeather|Sunrise|Sunset|
+-----------+--------------------+--------+---------+-------------------+------------------------+-------------------+------------------------+------------------------------+------------------------------+------------------------------+------------------+------------+-------+------+
|USW00014734|NEWARK LIBERTY IN...| 40.6825| -74.1694|2024-01-01 23:51:00|                     2.2|                0.0|                    NULL|     

In [25]:
df_filtered.write.parquet(f"{base_path}/data/weather/cleaned_weather/")

In [26]:
spark.stop()